# Generate CO2 Storage Nodes JSON for China (24 Basins)

Produces three JSON files (min / mean / max) using **deep saline aquifer** injection rates only.  
Each file can be pasted as a node block into `nodes_1.json`.

**Unit logic:**  
The MACRO model has 12 subperiods × 24 hours = 288 representative hours/year.  
The `CO2StorageConstraint` sums `subperiod_weight × flow_t` (t/hr × hrs = tonnes) per subperiod.  
`rhs_policy = saline_rate_Mt_yr × 1e6 / NUM_SUBPERIODS` (tonnes per subperiod).

In [ ]:
import pandas as pd
import json
import os

NUM_SUBPERIODS = 12

BASIN_KEY_MAP = {
    'Songliao Basin':             'Songliao',
    'Tuepan-Hami Basin':          'TurpanHami',   # typo in source CSV
    'Subei Basin':                'Subei',
    'Bohai Bay Basin (onshore)':  'BohaiOnshore',
    'Qaidam Basin':               'Qaidam',
    'Nanxiang Basin':             'Nanxiang',
    'Sanjiang Basin':             'Sanjiang',
    'Hailar Basin':               'Hailar',
    'Jianghan Basin':             'Jianghan',
    'Tarim Basin':                'Tarim',
    'Ordos Basin':                'Ordos',
    'Ejinjina Basin':             'YingenEjina',  # alt transliteration
    'Hehuai Basin':               'Hehuai',
    'Qinshui Basin':              'Qinshui',
    'Erlian Basin':               'Erlian',
    'Junggar Basin':              'Junggar',
    'Sichuan Basin':              'SichuanBasin',
    'Bohai Bay Basin (offshore)': 'BohaiOffshore',
    'North Yellow Sea Basin':     'NorthYellowSea',
    'South Yellow Sea Basin':     'SouthYellowSea',
    'East China Sea Basin':       'EastChinaSea',
    'Pearl River Mouth Basin':    'PearlRiverMouth',
    'Beibu Gulf Basin':           'BeibugGulf',
    'Qiongdongnan Basin':         'Qiongdongnan',
}

# Saline aquifer columns only
SALINE_COLS = {
    'min':  'Saline Aquifer Min (Mt/a)',
    'mean': 'Saline Aquifer Mean (Mt/a)',
    'max':  'Saline Aquifer Max (Mt/a)',
}

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('make_co2_storage_nodes.ipynb'))
INJECTION_CSV = os.path.join(NOTEBOOK_DIR, 'nature_scientific_data_source', 'injection_rates.csv')

df = pd.read_csv(INJECTION_CSV)

# Parse all saline columns to numeric
for col in SALINE_COLS.values():
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Map basin names
df['basin_key'] = df['Basin'].map(BASIN_KEY_MAP)
unmatched = df[df['basin_key'].isna()]['Basin'].tolist()
if unmatched:
    print(f"WARNING: unmatched basin names: {unmatched}")

df = df[df['basin_key'].notna()].copy()
print(f"Loaded {len(df)} basins")
print(df[['Basin', 'basin_key'] + list(SALINE_COLS.values())].to_string(index=False))

In [ ]:
def make_node_section(df, saline_col):
    """Build the CO2Captured node dict for one estimate scenario."""
    instance_data = []
    for _, row in df.iterrows():
        rhs = int(round(row[saline_col] * 1e6 / NUM_SUBPERIODS))
        instance_data.append({
            "id": f"co2_storage_{row['basin_key']}",
            "constraints": {"CO2StorageConstraint": True},
            "rhs_policy": {"CO2StorageConstraint": rhs}
        })
    return {
        "type": "CO2Captured",
        "global_data": {
            "time_interval": "CO2Captured",
            "constraints": {"BalanceConstraint": False}
        },
        "instance_data": instance_data
    }


# Generate and write all three files
output_files = {}
for scenario, saline_col in SALINE_COLS.items():
    section = make_node_section(df, saline_col)
    fname = f"co2_storage_nodes_{scenario}.json"
    out_path = os.path.join(NOTEBOOK_DIR, fname)
    with open(out_path, 'w') as f:
        json.dump(section, f, indent=2)
    output_files[scenario] = out_path
    rhs_vals = [inst['rhs_policy']['CO2StorageConstraint'] for inst in section['instance_data']]
    print(f"{scenario:4s}  -> {fname}")
    print(f"       rhs_policy range: {min(rhs_vals):>12,.0f} – {max(rhs_vals):>12,.0f} t/subperiod")
    print(f"       total saline capacity: {df[saline_col].sum():.1f} Mt/yr across all basins")
    print()

In [ ]:
# Cross-check: all node IDs must match co2_injection.csv end_vertices
inj_csv = os.path.join(NOTEBOOK_DIR, 'co2_injection.csv')
if os.path.exists(inj_csv):
    inj_df = pd.read_csv(inj_csv)
    storage_vtx = set(inj_df['edges--co2_storage_edge--end_vertex'].unique())
    node_ids    = set(f"co2_storage_{row['basin_key']}" for _, row in df.iterrows())
    missing_from_nodes = storage_vtx - node_ids
    missing_from_inj   = node_ids - storage_vtx
    if missing_from_nodes:
        print(f"WARNING: injection.csv references vertices not in storage nodes: {missing_from_nodes}")
    if missing_from_inj:
        print(f"WARNING: storage nodes not referenced in injection.csv: {missing_from_inj}")
    if not missing_from_nodes and not missing_from_inj:
        print("Cross-check PASSED: all 24 node IDs match co2_injection.csv end_vertices.")

# Preview mean file
print("\n--- Preview: co2_storage_nodes_mean.json (first 3 nodes) ---")
with open(output_files['mean']) as f:
    mean_json = json.load(f)
preview = {**mean_json, 'instance_data': mean_json['instance_data'][:3]}
print(json.dumps(preview, indent=2))